# **3D Reconstruction MASt3R w/ps2** 



**MASt3R is a learning-based, SfM-free 3D reconstruction method that directly regresses dense 3D geometry from image pairs, eliminating the need for feature matching, triangulation, and bundle adjustment.**




---

### Pipeline Overview

This pipeline performs **end-to-end 3D reconstruction** from a set of images, replacing the traditional COLMAP SfM pipeline with **DINO + MASt3R**, and finally exporting results back into **COLMAP format** for downstream use (e.g. Gaussian Splatting).

---

### 1. Image Preprocessing (Biplet-Square Normalization)

* Each input image is converted into **two square crops** (left/right or top/bottom).
* This normalizes aspect ratios and increases overlap robustness without resizing distortions.

---

### 2. Image Pair Selection (DINO Global Features)

* DINOv2 is used to extract **global image descriptors**.
* Images are paired based on **top-K cosine similarity**, with a diversity-aware selection strategy.
* This drastically reduces the number of pairs while preserving scene coverage and saving memory.

---

### 3. 3D Reconstruction with MASt3R

* MASt3R replaces traditional keypoint detection, matching, and SfM.
* Selected image pairs are processed to predict **dense correspondences and 3D structure**.
* A global alignment step optimizes camera poses and point clouds into a coherent scene.

---

### 4. Conversion to COLMAP Format

* The MASt3R scene is converted into **COLMAP-compatible outputs**:

  * `cameras.bin`, `images.bin`, `points3D.bin`
  * RGB images, depth maps, normal maps, and confidence masks
* This enables compatibility with existing COLMAP-based tools and pipelines.

---

### 5. Visualization

* The reconstructed 3D points are exported as a **PLY point cloud**.
* Open3D is used for lightweight, CPU-based visualization.

---

### Key Characteristics

* **Learning-based SfM** (no feature matching or bundle adjustment)
* **Memory-aware design** (pair limiting, reduced resolution, aggressive cleanup)
* **Drop-in replacement** for COLMAP reconstruction in modern pipelines
* **Ready for Gaussian Splatting or neural rendering**

---


# Setup

In [ ]:
import os
import sys
import gc
import h5py
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from pathlib import Path
import subprocess
from PIL import Image, ImageFilter
import struct

# Transformers for DINO
from transformers import AutoImageProcessor, AutoModel

In [ ]:
class Config:
    # Feature extraction
    N_KEYPOINTS = 4096
    IMAGE_SIZE = 512

    # Pair selection - CRITICAL for memory
    GLOBAL_TOPK = 20
    MIN_MATCHES = 10
    RATIO_THR = 1.2

    # Paths
    DINO_MODEL = "facebook/dinov2-base"

    MAST3R_MODEL = "/kaggle/working/mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth"
    MAST3R_IMAGE_SIZE = 256  # should be %16=0

    # Device
    DEVICE = torch.device('cpu')

In [ ]:
# ============================================================================
# Memory Management Utilities
# ============================================================================

def clear_memory():
    """Aggressively clear GPU and CPU memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def get_memory_info():
    """Get current memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory - Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")
    
    import psutil
    cpu_mem = psutil.virtual_memory().percent
    print(f"CPU Memory Usage: {cpu_mem:.1f}%")

# ============================================================================
# Environment Setup
# ============================================================================

def run_cmd(cmd, check=True, capture=False):
    """Run command with better error handling"""
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(
        cmd,
        capture_output=capture,
        text=True,
        check=False
    )
    if check and result.returncode != 0:
        print(f"❌ Command failed with code {result.returncode}")
        if capture:
            print(f"STDOUT: {result.stdout}")
            print(f"STDERR: {result.stderr}")
    return result


def setup_base_environment():
    """Setup base Python environment"""
    print("\n=== Setting up Base Environment ===")
    
    # NumPy fix for Python 3.12
    print("\n📦 Fixing NumPy...")
    run_cmd([sys.executable, "-m", "pip", "uninstall", "-y", "numpy"])
    run_cmd([sys.executable, "-m", "pip", "install", "numpy==1.26.4"])
    
    # PyTorch
    print("\n📦 Installing PyTorch...")
    run_cmd([
        sys.executable, "-m", "pip", "install",
        "torch", "torchvision", "torchaudio"
    ])
    
    # Core utilities
    print("\n📦 Installing core utilities...")
    run_cmd([
        sys.executable, "-m", "pip", "install",
        "opencv-python",
        "pillow",
        "imageio",
        "imageio-ffmpeg",
        "plyfile",
        "tqdm",
        "tensorboard",
        "scipy",  # for rotation conversions and image resizing
        "psutil"  # for memory monitoring
    ])
    
    # Transformers for DINO
    print("\n📦 Installing transformers...")
    run_cmd([
        sys.executable, "-m", "pip", "install",
        "transformers==4.40.0"
    ])
    
    # pycolmap for COLMAP format
    print("\n📦 Installing pycolmap...")
    run_cmd([sys.executable, "-m", "pip", "install", "pycolmap"])
    
    print("✓ Base environment setup complete!")


def setup_mast3r():
    """Install and setup MASt3R"""
    print("\n=== Setting up MASt3R ===")
    
    os.chdir('/kaggle/working')
    
    # Remove existing installation
    if os.path.exists('mast3r'):
        print("Removing existing MASt3R installation...")
        os.system('rm -rf mast3r')
    
    # Clone repository
    print("Cloning MASt3R repository...")
    os.system('git clone --recursive https://github.com/naver/mast3r')
    os.chdir('/kaggle/working/mast3r')
    
    # Check dust3r directory
    print("Checking dust3r structure...")
    os.system('ls -la dust3r/')
    
    # Install dust3r
    print("Installing dust3r...")
    os.system('cd dust3r && python -m pip install -e .')
    
    # Install croco
    print("Installing croco...")
    os.system('cd dust3r/croco && python -m pip install -e .')
    
    # Install requirements
    print("Installing MASt3R requirements...")
    os.system('pip install -r requirements.txt')
    
    # Download model weights
    print("Downloading model weights...")
    os.system('mkdir -p checkpoints')
    os.system('wget -P checkpoints/ https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth')
    
    # Install additional dependencies
    print("Installing additional dependencies...")
    os.system('pip install trimesh matplotlib roma')
    
    # Add to path
    sys.path.insert(0, '/kaggle/working/mast3r')
    sys.path.insert(0, '/kaggle/working/mast3r/dust3r')
    
    # Verification
    print("\n🔍 Verifying MASt3R installation...")
    try:
        from mast3r.model import AsymmetricMASt3R
        print("  ✓ MASt3R import: OK")
    except Exception as e:
        print(f"  ❌ MASt3R import failed: {e}")
        raise
    
    print("✓ MASt3R setup complete!")

def setup_gaussian_splatting():
    """Setup Gaussian Splatting"""
    print("\n=== Setting up Gaussian Splatting ===")
    
    os.chdir('/kaggle/working')
    
    WORK_DIR = "gaussian-splatting"
    
    if not os.path.exists(WORK_DIR):
        print("Cloning Gaussian Splatting repository...")
        run_cmd([
            "git", "clone", "--recursive",
            "https://github.com/graphdeco-inria/gaussian-splatting.git",
            WORK_DIR
        ])
    else:
        print("✓ Repository already exists")
    
    os.chdir(WORK_DIR)
    
    # Install requirements
    print("Installing Gaussian Splatting requirements...")
    run_cmd([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
    
    # Build submodules
    print("\n📦 Building Gaussian Splatting submodules...")
    
    submodules = {
        "diff-gaussian-rasterization":
            "https://github.com/graphdeco-inria/diff-gaussian-rasterization.git",
        "simple-knn":
            "https://github.com/camenduru/simple-knn.git"
    }
    
    for name, repo in submodules.items():
        print(f"\n📦 Installing {name}...")
        path = os.path.join("submodules", name)
        if not os.path.exists(path):
            run_cmd(["git", "clone", repo, path])
        run_cmd([sys.executable, "-m", "pip", "install", path])
    
    print("✓ Gaussian Splatting setup complete!")

In [ ]:
setup_base_environment()
clear_memory()

setup_mast3r()
clear_memory()


In [ ]:
# ============================================================================
# Step 0: Biplet-Square Normalization (PRESERVED FROM ORIGINAL)
# ============================================================================

def normalize_image_sizes_biplet(input_dir, output_dir=None, size=1024):
    """
    Generates two square crops (Left & Right or Top & Bottom)
    from each image in a directory.
    """
    if output_dir is None:
        output_dir = 'output/images_biplet'

    os.makedirs(output_dir, exist_ok=True)

    print(f"Generating 2 cropped squares (Left/Right or Top/Bottom) for each image...")
    print()

    converted_count = 0
    size_stats = {}

    for img_file in sorted(os.listdir(input_dir)):
        if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        input_path = os.path.join(input_dir, img_file)

        try:
            img = Image.open(input_path)
            original_size = img.size

            size_key = f"{original_size[0]}x{original_size[1]}"
            size_stats[size_key] = size_stats.get(size_key, 0) + 1

            # Generate 2 crops
            crops = generate_two_crops(img, size)

            base_name, ext = os.path.splitext(img_file)
            for mode, cropped_img in crops.items():
                output_path = os.path.join(output_dir, f"{base_name}_{mode}{ext}")
                cropped_img.save(output_path, quality=95)

            converted_count += 1
            print(f"  ✓ {img_file}: {original_size} → 2 square images generated")

        except Exception as e:
            print(f"  ✗ Error processing {img_file}: {e}")

    print(f"\nProcessing complete: {converted_count} source images processed")
    print(f"Original size distribution: {size_stats}")
    return converted_count

In [ ]:
def generate_two_crops(img, size):
    """
    Generates two square crops from an image.
    """
    # If size is a tuple or list, extract the first value
    if isinstance(size, (tuple, list)):
        size = size[0]
    
    width, height = img.size
    crops = {}
    
    if width >= height:
        # Landscape: Split into left and right squares
        box_left = (0, 0, height, height)
        box_right = (width - height, 0, width, height)
        crops['left'] = img.crop(box_left).resize((size, size), Image.LANCZOS)
        crops['right'] = img.crop(box_right).resize((size, size), Image.LANCZOS)
    else:
        # Portrait: Split into top and bottom squares
        box_top = (0, 0, width, width)
        box_bottom = (0, height - width, width, height)
        crops['top'] = img.crop(box_top).resize((size, size), Image.LANCZOS)
        crops['bottom'] = img.crop(box_bottom).resize((size, size), Image.LANCZOS)
    
    return crops

In [ ]:
# ============================================================================
# Step 1: DINO-based Pair Selection (PRESERVED FROM ORIGINAL)
# ============================================================================

def load_torch_image(fname, device):
    """Load image as torch tensor"""
    import torchvision.transforms as T

    img = Image.open(fname).convert('RGB')
    transform = T.Compose([
        T.ToTensor(),
    ])
    return transform(img).unsqueeze(0).to(device)

def extract_dino_global(image_paths, model_path, device):
    """Extract DINO global descriptors with memory management"""
    print("\n=== Extracting DINO Global Features ===")
    print("Initial memory state:")
    get_memory_info()

    processor = AutoImageProcessor.from_pretrained(model_path)
    model = AutoModel.from_pretrained(model_path).eval().to(device)

    global_descs = []
    batch_size = 4  # Small batch to save memory
    
    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch_paths = image_paths[i:i+batch_size]
        batch_imgs = []
        
        for img_path in batch_paths:
            img = load_torch_image(img_path, device)
            batch_imgs.append(img)
        
        batch_tensor = torch.cat(batch_imgs, dim=0)
        
        with torch.no_grad():
            inputs = processor(images=batch_tensor, return_tensors="pt", do_rescale=False).to(device)
            outputs = model(**inputs)
            desc = F.normalize(outputs.last_hidden_state[:, 1:].max(dim=1)[0], dim=1, p=2)
            global_descs.append(desc.cpu())
        
        # Clear batch memory
        del batch_tensor, inputs, outputs, desc
        clear_memory()

    global_descs = torch.cat(global_descs, dim=0)

    del model, processor
    clear_memory()
    
    print("After DINO extraction:")
    get_memory_info()

    return global_descs

def build_topk_pairs(global_feats, k, device):
    """Build top-k similar pairs from global features"""
    g = global_feats.to(device)
    sim = g @ g.T
    sim.fill_diagonal_(-1)

    N = sim.size(0)
    k = min(k, N - 1)

    topk_indices = torch.topk(sim, k, dim=1).indices.cpu()

    pairs = []
    for i in range(N):
        for j in topk_indices[i]:
            j = j.item()
            if i < j:
                pairs.append((i, j))

    # Remove duplicates
    pairs = list(set(pairs))
    
    return pairs

def select_diverse_pairs(pairs, max_pairs, num_images):
    """
    Select diverse pairs to ensure good image coverage
    Strategy: Select pairs that maximize image coverage
    """
    import random
    random.seed(42)
    
    if len(pairs) <= max_pairs:
        return pairs
    
    print(f"Selecting {max_pairs} diverse pairs from {len(pairs)} candidates...")
    
    # Count how many times each image appears in pairs
    image_counts = {i: 0 for i in range(num_images)}
    for i, j in pairs:
        image_counts[i] += 1
        image_counts[j] += 1
    
    # Sort pairs by: prefer pairs with less-connected images
    def pair_score(pair):
        i, j = pair
        # Lower score = images appear in fewer pairs = more diverse
        return image_counts[i] + image_counts[j]
    
    pairs_scored = [(pair, pair_score(pair)) for pair in pairs]
    pairs_scored.sort(key=lambda x: x[1])
    
    # Select pairs greedily to maximize coverage
    selected = []
    selected_images = set()
    
    # Phase 1: Select pairs that add new images (greedy coverage)
    for pair, score in pairs_scored:
        if len(selected) >= max_pairs:
            break
        i, j = pair
        # Prefer pairs that include new images
        if i not in selected_images or j not in selected_images:
            selected.append(pair)
            selected_images.add(i)
            selected_images.add(j)
    
    # Phase 2: Fill remaining slots with high-similarity pairs
    if len(selected) < max_pairs:
        remaining = [p for p, s in pairs_scored if p not in selected]
        random.shuffle(remaining)
        selected.extend(remaining[:max_pairs - len(selected)])
    
    print(f"Selected pairs cover {len(selected_images)} / {num_images} images ({100*len(selected_images)/num_images:.1f}%)")
    
    return selected

def get_image_pairs_dino(image_paths, max_pairs=None):
    """DINO-based pair selection with intelligent limiting"""
    device = Config.DEVICE

    # DINO global features
    global_feats = extract_dino_global(image_paths, Config.DINO_MODEL, device)
    pairs = build_topk_pairs(global_feats, Config.GLOBAL_TOPK, device)

    print(f"Initial pairs from DINO: {len(pairs)}")
    
    # Apply intelligent pair selection if limit specified
    if max_pairs and len(pairs) > max_pairs:
        pairs = select_diverse_pairs(pairs, max_pairs, len(image_paths))
    
    return pairs

# ============================================================================
# Step 2: MASt3R Reconstruction (REPLACES ALIKED/LIGHTGLUE/COLMAP)
# ============================================================================

def load_mast3r_model(device='cuda'):
    """Load MASt3R model"""
    from mast3r.model import AsymmetricMASt3R
    
    model = AsymmetricMASt3R.from_pretrained(Config.MAST3R_MODEL).to(device)
    model.eval()
    
    print(f"✓ MASt3R model loaded on {device}")
    return model

def load_images_for_mast3r(image_paths, size=224):
    """Load images using DUSt3R's format with reduced size"""
    print(f"\n=== Loading images for MASt3R (size={size}) ===")
    
    from dust3r.utils.image import load_images
    
    # Load images using DUSt3R's loader with reduced size
    images = load_images(image_paths, size=size, verbose=True)
    
    return images

def run_mast3r_pairs(model, image_paths, pairs, device='cuda', batch_size=1, max_pairs=None):
    """Run MASt3R on selected pairs with memory management"""
    print("\n=== Running MASt3R Reconstruction ===")
    print("Initial memory state:")
    get_memory_info()
    
    from dust3r.inference import inference
    from dust3r.cloud_opt import global_aligner, GlobalAlignerMode
    
    # Limit number of pairs if specified
    if max_pairs and len(pairs) > max_pairs:
        print(f"Limiting pairs from {len(pairs)} to {max_pairs}")
        # Select pairs more evenly distributed
        step = max(1, len(pairs) // max_pairs)
        pairs = pairs[::step][:max_pairs]
    
    print(f"Processing {len(pairs)} pairs...")
    
    # Load images in smaller size
    print(f"Loading {len(image_paths)} images at {Config.MAST3R_IMAGE_SIZE}x{Config.MAST3R_IMAGE_SIZE}...")
    images = load_images_for_mast3r(image_paths, size=Config.MAST3R_IMAGE_SIZE)
    
    print(f"Loaded {len(images)} images")
    print("After loading images:")
    get_memory_info()
    
    # Create all image pairs at once
    print(f"Creating {len(pairs)} image pairs...")
    mast3r_pairs = []
    for idx1, idx2 in tqdm(pairs, desc="Preparing pairs"):
        mast3r_pairs.append((images[idx1], images[idx2]))
    
    print(f"Running MASt3R inference on {len(mast3r_pairs)} pairs...")
    
    # Run inference (this returns the dict format we need)
    output = inference(mast3r_pairs, model, device, batch_size=batch_size, verbose=True)
    
    # Clear pairs from memory
    del mast3r_pairs
    clear_memory()
    
    print("✓ MASt3R inference complete")
    print("After inference:")
    get_memory_info()
    
    # Global alignment
    print("Running global alignment...")
    scene = global_aligner(
        output, 
        device=device, 
        mode=GlobalAlignerMode.PointCloudOptimizer
    )
    
    # Clear output after creating scene
    del output
    clear_memory()
    
    print("Computing global alignment...")
    loss = scene.compute_global_alignment(
        init="mst", 
        niter=150,  # Reduced from 300
        schedule='cosine', 
        lr=0.01
    )
    
    print(f"✓ Global alignment complete (final loss: {loss:.6f})")
    print("Final memory state:")
    get_memory_info()
    
    return scene, images

# Process2

## **The MASt3R scene is converted into COLMAP-compatible outputs.**

In [ ]:
#process2_15.py
import struct
import numpy as np
from pathlib import Path
from PIL import Image
import os
import torch

def rotmat_to_qvec(R):
    """Convert Rotation Matrix to Quaternion (w, x, y, z)"""
    R = np.asarray(R, dtype=np.float64)
    trace = np.trace(R)
    if trace > 0:
        s = 0.5 / np.sqrt(trace + 1.0)
        w, x, y, z = 0.25 / s, (R[2, 1] - R[1, 2]) * s, (R[0, 2] - R[2, 0]) * s, (R[1, 0] - R[0, 1]) * s
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        s = 2.0 * np.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2])
        w, x, y, z = (R[2, 1] - R[1, 2]) / s, 0.25 * s, (R[0, 1] + R[1, 0]) / s, (R[0, 2] + R[2, 0]) / s
    elif R[1, 1] > R[2, 2]:
        s = 2.0 * np.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2])
        w, x, y, z = (R[0, 2] - R[2, 0]) / s, (R[0, 1] + R[1, 0]) / s, 0.25 * s, (R[1, 2] + R[2, 1]) / s
    else:
        s = 2.0 * np.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1])
        w, x, y, z = (R[1, 0] - R[0, 1]) / s, (R[0, 2] + R[2, 0]) / s, (R[1, 2] + R[2, 1]) / s, 0.25 * s
    q = np.array([w, x, y, z])
    return q / np.linalg.norm(q)

def extract_all_data(scene, image_paths, conf_threshold=1.5):
    """
    Extract 3D points, colors, confidence, and camera parameters from the scene.
    """
    print("\n=== Extracting Data from Scene ===")
    
    # Basic data from scene
    pts3d_list = scene.get_pts3d()
    im_poses = scene.get_im_poses().detach().cpu().numpy()
    focals = scene.get_focals().detach().cpu().numpy()
    pp = scene.get_principal_points().detach().cpu().numpy()
    im_conf = scene.im_conf

    all_pts, all_cols, all_conf = [], [], []
    cameras_dict = {}

    for i, img_path in enumerate(image_paths):
        img_name = os.path.basename(img_path)
        img_raw = Image.open(img_path).convert('RGB')
        W_orig, H_orig = img_raw.size
        
        # Get 3D points and confidence
        pts = pts3d_list[i].detach().cpu().numpy()
        conf = im_conf[i].detach().cpu().numpy()
        H_pts, W_pts = pts.shape[:2]

        # Sample colors at the same resolution as 3D points
        img_res = img_raw.resize((W_pts, H_pts), Image.BILINEAR)
        cols = np.array(img_res)

        # Filter by confidence threshold
        mask = conf > conf_threshold
        all_pts.append(pts[mask])
        all_cols.append(cols[mask])
        all_conf.append(conf[mask])

        # Camera parameter calculation with scale correction
        scale = W_orig / W_pts
        fx = float(focals[i, 0] if focals.ndim > 1 else focals[i]) * scale
        fy = float(focals[i, 1] if (focals.ndim > 1 and focals.shape[1] > 1) else (focals[i, 0] if focals.ndim > 1 else focals[i])) * scale
        cx, cy = pp[i, 0] * scale, pp[i, 1] * scale

        cameras_dict[img_name] = {
            'id': i + 1,
            'w': W_orig, 'h': H_orig,
            'params': (fx, fy, cx, cy),
            'pose_w2c': np.linalg.inv(im_poses[i]) # World-to-Camera
        }
        print(f"  Image {i+1}: {img_name} ({len(pts[mask]):,} points extracted)")

    return (np.concatenate(all_pts), np.concatenate(all_cols), 
            np.concatenate(all_conf), cameras_dict)

def save_colmap_binary(pts3d, colors, conf, cameras_dict, output_dir):
    """
    Save data in COLMAP binary format (.bin).
    """
    path = Path(output_dir)
    path.mkdir(parents=True, exist_ok=True)

    # 1. cameras.bin (PINHOLE model)
    with open(path / 'cameras.bin', 'wb') as f:
        f.write(struct.pack('Q', len(cameras_dict)))
        for name, cam in cameras_dict.items():
            f.write(struct.pack('IiQQ', cam['id'], 1, cam['w'], cam['h']))
            f.write(struct.pack('dddd', *cam['params']))

    # 2. images.bin
    with open(path / 'images.bin', 'wb') as f:
        f.write(struct.pack('Q', len(cameras_dict)))
        for name, cam in cameras_dict.items():
            q = rotmat_to_qvec(cam['pose_w2c'][:3, :3])
            t = cam['pose_w2c'][:3, 3]
            f.write(struct.pack('I', cam['id']))
            f.write(struct.pack('dddd', *q))
            f.write(struct.pack('ddd', *t))
            f.write(struct.pack('I', cam['id']))
            f.write(name.encode('utf-8') + b'\x00')
            f.write(struct.pack('Q', 0))

    # 3. points3D.bin (Colored)
    with open(path / 'points3D.bin', 'wb') as f:
        f.write(struct.pack('Q', len(pts3d)))
        for i, (pt, col, cf) in enumerate(zip(pts3d, colors, conf)):
            f.write(struct.pack('Q', i + 1))
            f.write(struct.pack('ddd', *pt))
            f.write(struct.pack('BBB', *col)) # RGB
            f.write(struct.pack('d', 1.0 / max(cf, 0.01))) # Error estimate
            f.write(struct.pack('Q', 0))

    print(f"\n✓ COLMAP binary files exported to: {output_dir}")

def write_colored_ply(pts3d, colors, output_path):
    """Export a colored PLY file for visualization."""
    with open(output_path, 'w') as f:
        f.write(f"ply\nformat ascii 1.0\nelement vertex {len(pts3d)}\n"
                "property float x\nproperty float y\nproperty float z\n"
                "property uchar red\nproperty uchar green\nproperty uchar blue\n"
                "end_header\n")
        for pt, col in zip(pts3d, colors):
            f.write(f"{pt[0]} {pt[1]} {pt[2]} {int(col[0])} {int(col[1])} {int(col[2])}\n")
    print(f"✓ PLY file saved: {output_path}")

def create_colmap_bins(scene, image_paths, output_dir, conf_threshold=1.5):
    """
    Main entry point to generate COLMAP reconstruction data.
    """
    print("="*60)
    print("DUST3R TO COLMAP RECONSTRUCTION")
    print("="*60)

    # 1. Extraction
    pts3d, colors, conf, cameras_dict = extract_all_data(scene, image_paths, conf_threshold)

    # 2. COLMAP Binary Output
    save_colmap_binary(pts3d, colors, conf, cameras_dict, output_dir)

    # 3. PLY Output (Optional verification)
    write_colored_ply(pts3d, colors, Path(output_dir) / "point_cloud.ply")

    print("\n✓ PROCESS COMPLETE")
    return cameras_dict, pts3d, conf, colors




# bins to ply

In [ ]:
def convert_colmap_bin_to_ply(sparse_dir, output_ply_path):

    import pycolmap
    from plyfile import PlyData, PlyElement
    import numpy as np
    
    print(f"\n=== Converting COLMAP bin to PLY ===")

    reconstruction = pycolmap.Reconstruction(str(sparse_dir))
    
    print(f"Loaded reconstruction:")
    print(f"  - {len(reconstruction.cameras)} cameras")
    print(f"  - {len(reconstruction.images)} images")
    print(f"  - {len(reconstruction.points3D)} points")
    
    if len(reconstruction.points3D) == 0:
        print("❌ No 3D points in reconstruction!")
        return 0

    points = []
    colors = []
    
    for point3D_id, point3D in reconstruction.points3D.items():
        points.append(point3D.xyz)
        colors.append(point3D.color)
    
    points = np.array(points)
    colors = np.array(colors)
    
    print(f"\nPoint cloud statistics:")
    print(f"  Total points: {len(points)}")
    print(f"  X range: [{points[:, 0].min():.3f}, {points[:, 0].max():.3f}]")
    print(f"  Y range: [{points[:, 1].min():.3f}, {points[:, 1].max():.3f}]")
    print(f"  Z range: [{points[:, 2].min():.3f}, {points[:, 2].max():.3f}]")

    vertices = np.array(
        [(p[0], p[1], p[2], c[0], c[1], c[2]) 
         for p, c in zip(points, colors)],
        dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4'),
               ('red', 'u1'), ('green', 'u1'), ('blue', 'u1')]
    )
    
    el = PlyElement.describe(vertices, 'vertex')
    PlyData([el]).write(output_ply_path)
    
    print(f"✓ Saved PLY file to {output_ply_path}")
    
    return len(points)


'''
colmap_output_dir='/kaggle/working/output/colmap'
sparse_dir = os.path.join(colmap_output_dir, 'sparse', '0')
ply_path = os.path.join(colmap_output_dir, 'point_cloud.ply')
num_points = convert_colmap_bin_to_ply(sparse_dir, ply_path)
'''


# main_pipeline

In [ ]:
# --- Keeping the initial image processing and model inference as is ---
image_dir = "/kaggle/input/image-matching-challenge-2023/train/haiper/bike/images"
output_dir = "/kaggle/working/output"
square_size = 1024 
max_pairs = 1000   
max_points = 1000000   
os.makedirs(output_dir, exist_ok=True)
processed_image_dir = os.path.join(output_dir, "processed_images")

# Image normalization
normalize_image_sizes_biplet(
    input_dir=image_dir,
    output_dir=processed_image_dir,
    size=square_size
)

processed_image_paths = sorted([
    os.path.join(processed_image_dir, f) 
    for f in os.listdir(processed_image_dir) 
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

# Create pairs and execute MASt3R
pairs = get_image_pairs_dino(processed_image_paths, max_pairs=max_pairs)
clear_memory()

device = Config.DEVICE
model = load_mast3r_model(device)
scene, mast3r_images = run_mast3r_pairs(
    model, processed_image_paths, pairs, device,
    max_pairs=None
)

# Release model to free up memory
del model
clear_memory()

# ========================================================
# REVISED: Processing everything at once with new functions
# ========================================================

# 1. Set output directory (COLMAP standard sparse/0 format)
colmap_sparse_dir = os.path.join(output_dir, "colmap/sparse/0")

# 2. Extract coordinates, colors, and camera params, then save COLMAP binaries
# This function handles color extraction (matching coordinates) and BIN file export simultaneously
cameras_dict, pts3d, confidence, colors = create_colmap_bins(
    scene=scene, 
    image_paths=processed_image_paths, 
    output_dir=colmap_sparse_dir, 
    conf_threshold=1.5
)

# 3. Finally, delete the scene object and clear memory
del scene
clear_memory()

# 4. Verify results (The PLY file is also saved within create_colmap_bins)
print(f"✓ COLMAP binary files and colored PLY created at: {colmap_sparse_dir}")
print(f"✓ Total points processed: {len(pts3d):,}")

# Ply Viewer

In [ ]:
!pip install open3d

In [ ]:
"""
PLY Viewer for Kaggle Notebook
Display PLY files from /kaggle/working directory
"""

from IPython.display import HTML, display
import base64

def display_ply_viewer(ply_file_path):
    """
    Display PLY file in Kaggle notebook
    
    Args:
        ply_file_path: Path to PLY file (e.g., '/kaggle/working/model.ply')
    """
    
    # Read PLY file and encode to Base64
    with open(ply_file_path, 'rb') as f:
        ply_data = f.read()
        ply_base64 = base64.b64encode(ply_data).decode('utf-8')
    
    # Create HTML viewer
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>PLY Viewer</title>
        <style>
            body {{ margin: 0; font-family: Arial, sans-serif; }}
            #container {{ width: 100%; height: 600px; position: relative; }}
            #info {{
                position: absolute;
                top: 10px;
                left: 10px;
                background: rgba(0,0,0,0.7);
                color: white;
                padding: 10px;
                border-radius: 5px;
                font-size: 12px;
                z-index: 100;
            }}
            .controls {{
                position: absolute;
                top: 10px;
                right: 10px;
                background: rgba(0,0,0,0.7);
                padding: 10px;
                border-radius: 5px;
                z-index: 100;
            }}
            .button {{
                background: #4CAF50;
                color: white;
                border: none;
                padding: 8px 12px;
                margin: 2px;
                border-radius: 3px;
                cursor: pointer;
                font-size: 13px;
            }}
            .button:hover {{
                background: #45a049;
            }}
        </style>
    </head>
    <body>
        <div id="container">
            <div id="info">Loading...</div>
            <div class="controls">
                <button id="reset-view" class="button">Reset View</button>
            </div>
        </div>

        <script type="importmap">
        {{
          "imports": {{
            "three": "https://unpkg.com/three@0.160.0/build/three.module.js",
            "three/examples/jsm/loaders/PLYLoader.js": "https://unpkg.com/three@0.160.0/examples/jsm/loaders/PLYLoader.js",
            "three/examples/jsm/controls/OrbitControls.js": "https://unpkg.com/three@0.160.0/examples/jsm/controls/OrbitControls.js"
          }}
        }}
        </script>
        <script type="module">
            import * as THREE from 'three';
            import {{ PLYLoader }} from 'three/examples/jsm/loaders/PLYLoader.js';
            import {{ OrbitControls }} from 'three/examples/jsm/controls/OrbitControls.js';

            let scene, camera, renderer, controls;
            let currentPointCloud = null;

            function init() {{
                const container = document.getElementById('container');
                
                scene = new THREE.Scene();
                scene.background = new THREE.Color(0x1a1a1a);

                camera = new THREE.PerspectiveCamera(60, container.clientWidth / container.clientHeight, 0.1, 10000);
                camera.position.set(5, 5, 10);

                renderer = new THREE.WebGLRenderer({{ antialias: true }});
                renderer.setSize(container.clientWidth, container.clientHeight);
                container.appendChild(renderer.domElement);

                controls = new OrbitControls(camera, renderer.domElement);
                controls.enableDamping = true;
                controls.dampingFactor = 0.05;

                const ambientLight = new THREE.AmbientLight(0xffffff, 0.6);
                scene.add(ambientLight);

                const directionalLight = new THREE.DirectionalLight(0xffffff, 0.8);
                directionalLight.position.set(10, 20, 15);
                scene.add(directionalLight);

                const gridHelper = new THREE.GridHelper(100, 50, 0x444444, 0x222222);
                scene.add(gridHelper);

                document.getElementById('reset-view').addEventListener('click', resetView);

                loadPLY();
                animate();
            }}

            function loadPLY() {{
                const plyBase64 = '{ply_base64}';
                const binaryString = atob(plyBase64);
                const bytes = new Uint8Array(binaryString.length);
                for (let i = 0; i < binaryString.length; i++) {{
                    bytes[i] = binaryString.charCodeAt(i);
                }}

                const loader = new PLYLoader();
                try {{
                    const geometry = loader.parse(bytes.buffer);
                    displayGeometry(geometry);
                }} catch (error) {{
                    document.getElementById('info').innerHTML = 'Error: ' + error.message;
                }}
            }}

            function displayGeometry(geometry) {{
                const hasColors = geometry.attributes.color !== undefined;
                
                // Fixed point size: 0.002
                const material = new THREE.PointsMaterial({{
                    size: 0.002,
                    vertexColors: hasColors,
                    sizeAttenuation: true
                }});

                if (!hasColors) {{
                    material.color = new THREE.Color(0x00aaff);
                }}

                const pointCloud = new THREE.Points(geometry, material);
                scene.add(pointCloud);
                currentPointCloud = pointCloud;

                geometry.computeBoundingBox();
                const bbox = geometry.boundingBox;
                const center = new THREE.Vector3();
                bbox.getCenter(center);
                const size = bbox.getSize(new THREE.Vector3());
                const maxDim = Math.max(size.x, size.y, size.z);

                const fov = camera.fov * (Math.PI / 180);
                let cameraDistance = Math.abs(maxDim / Math.tan(fov / 2)) * 1.5;
                
                camera.position.set(cameraDistance, cameraDistance * 0.7, cameraDistance);
                controls.target.copy(center);
                controls.update();

                const pointCount = geometry.attributes.position.count;
                document.getElementById('info').innerHTML = 
                    `<strong>Point Cloud</strong><br>
                     Points: ${{pointCount.toLocaleString()}}<br>
                     Size: (${{size.x.toFixed(2)}}, ${{size.y.toFixed(2)}}, ${{size.z.toFixed(2)}})`;
            }}

            function resetView() {{
                if (currentPointCloud) {{
                    currentPointCloud.geometry.computeBoundingBox();
                    const bbox = currentPointCloud.geometry.boundingBox;
                    const center = new THREE.Vector3();
                    bbox.getCenter(center);
                    const size = bbox.getSize(new THREE.Vector3());
                    const maxDim = Math.max(size.x, size.y, size.z);
                    
                    const fov = camera.fov * (Math.PI / 180);
                    let cameraDistance = Math.abs(maxDim / Math.tan(fov / 2)) * 1.5;
                    
                    camera.position.set(cameraDistance, cameraDistance * 0.7, cameraDistance);
                    controls.target.copy(center);
                    controls.update();
                }}
            }}

            function animate() {{
                requestAnimationFrame(animate);
                controls.update();
                renderer.render(scene, camera);
            }}

            init();
        </script>
    </body>
    </html>
    """
    
    display(HTML(html_content))



In [ ]:
# you can change position and zoom ratio.
display_ply_viewer('/kaggle/working/output/colmap/sparse/0/point_cloud.ply')

# Process Comparison

### Comparison Table: process1, 2, and 3

| Comparison Item | process1.py | process2.py | process3.py |
| --- | --- | --- | --- |
| **Intrinsics (Focal Length)** | Uses **scaled MASt3R estimates** | Uses **scaled MASt3R estimates** (supports iso/anisotropic) | **Simplified calculation** based on image size (`max(w, h) * 1.2`) |
| **Intrinsics (Principal Point)** | Uses **scaled MASt3R estimates** | Uses **scaled MASt3R estimates** | Fixed to the **image center** |
| **Camera Pose (Extrinsics)** | **Inverse matrix transform** of MASt3R poses (w2c) | **Inverse matrix transform** of MASt3R poses (w2c) | **Custom pose estimation** based on the median of 3D points |
| **3D Point Extraction & Filtering** | Random sampling (up to 1M pts) and NaN/Inf removal | Filtering based on **confidence threshold** (default 1.5) | **Confidence filtering** and sampling 10k points per image |
| **Color Information** | Colors from resized source images | Colors matching only filtered 3D points | Directly from scene image data |
| **Primary Output Files** | COLMAP sparse reconstruction binaries (cameras, images, points3D) | COLMAP sparse reconstruction binaries (with actual RGB colors) | Sparse binaries + **Depth and Normal maps** |
| **Main Use Case / Features** | Faithful conversion of MASt3R geometry to COLMAP | Emphasis on confidence filtering and accurate color extraction | Designed for integration with **COLMAP Dense Reconstruction** |

---

### Key Differences and Insights

1. **Camera Model Accuracy**
* **process1** and **process2** accurately reflect MASt3R's output by scaling the estimated focal length and principal point to the image dimensions.
* **process3** uses a simplified pinhole model where camera parameters are derived from a fixed formula.


2. **Pose Estimation Approach**
* **process1** and **process2** calculate the inverse of the "camera-to-world" matrix provided by MASt3R to fit the "world-to-camera" format required by COLMAP.
* **process3** employs a unique method of determining translation vectors based on the spatial distribution (median) of the generated 3D point cloud.


3. **Point Cloud Quality and Sampling**
* **process1** is designed to handle a high volume of points (up to 1 million).
* **process2** and **process3** prioritize data quality by using MASt3R's "confidence" scores to filter out unreliable points.


4. **Data Output Depth**
* **process3** provides the most comprehensive output, generating **depth maps and normal maps** in binary format. This makes it specifically optimized for COLMAP’s stereo pipeline, facilitating detailed 3D modeling beyond just sparse reconstruction.


https://www.kaggle.com/code/stpeteishii/3d-reconstruction-mast3r-w-ps1<br>
https://www.kaggle.com/code/stpeteishii/3d-reconstruction-mast3r-w-ps2<br>
https://www.kaggle.com/code/stpeteishii/3d-reconstruction-mast3r-w-ps3